In [1]:
import torch
import torch.nn as nn

import numpy as np

# 事前に保存しておいた embedding_matrix を読み込む
embedding_matrix = np.load('embedding_matrix.npy')  # shape=(vocab_size, emb_dim)


class AverageEmbeddingClassifier(nn.Module):
    def __init__(self, embedding_matrix):
        super().__init__()
        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float),
            freeze=True,          # ✅ 埋め込みを固定
            padding_idx=0
        )
        self.linear = nn.Linear(embedding_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, input_ids):
        embedded = self.embedding(input_ids)
        mask = (input_ids != 0).unsqueeze(-1)
        summed = (embedded * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1)
        mean_emb = summed / lengths
        return self.sigmoid(self.linear(mean_emb))

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class ReviewDataset(Dataset):
    def __init__(self, examples):
        self.data = examples

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

def collate_fn(batch):
    input_ids = [d['input_ids'] for d in batch]
    labels = torch.cat([d['label'] for d in batch])
    padded_ids = pad_sequence(input_ids, batch_first=True, padding_value=0)
    return padded_ids, labels

# モデル初期化（embedding_matrix はすでに用意済み）
model = AverageEmbeddingClassifier(embedding_matrix)

# ハイパーパラメータ
batch_size = 32
num_epochs = 5
lr = 1e-3

# データローダー
train_loader = DataLoader(
    ReviewDataset(train_data),
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

# 損失関数とOptimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Starting training...\n")

for epoch in range(1, num_epochs + 1):
    model.train()
    total_loss = 0

    for input_ids, labels in train_loader:
        input_ids = input_ids.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(input_ids).squeeze()
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch}: Loss = {avg_loss:.4f}")


NameError: name 'train_data' is not defined

In [2]:
import torch
import torch.nn as nn
import pickle
from torch.nn.utils.rnn import pad_sequence

# ロード
embedding_matrix = np.load('70_embeddings.npy')
with open('71_sst_data.pkl','rb') as f: data=pickle.load(f)
train_data=data['train']

# モデル
class BoWClassifier(nn.Module):
    def __init__(self, embedding_matrix):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix), freeze=True, padding_idx=0)
        emb_dim = embedding_matrix.shape[1]
        self.weight = nn.Parameter(torch.zeros(emb_dim))
        self.bias   = nn.Parameter(torch.zeros(1))
    def forward(self, input_ids):
        emb    = self.embedding(input_ids)
        mask   = (input_ids!=0).unsqueeze(-1).float()
        summed = (emb*mask).sum(dim=1)
        lengths= mask.sum(dim=1).clamp(min=1)
        avg    = summed/lengths
        logits = avg.matmul(self.weight) + self.bias
        return torch.sigmoid(logits)

model = BoWClassifier(embedding_matrix)
optimizer = torch.optim.Adam([model.weight, model.bias], lr=1e-3)
criterion = nn.BCELoss()

def make_batch(batch):
    x=pad_sequence([ex['input_ids'] for ex in batch], batch_first=True, padding_value=0)
    y=torch.cat([ex['label'] for ex in batch]).squeeze()
    return x,y

# 訓練ループ
model.train()
for epoch in range(5):
    for i in range(0,len(train_data),32):
        batch=train_data[i:i+32]
        x,y=make_batch(batch)
        optimizer.zero_grad()
        pred=model(x)
        loss=criterion(pred,y)
        loss.backward()
        optimizer.step()
        if i%320==0:
            print(f"Epoch{epoch} step{i} loss={loss.item():.4f}")
# 保存
torch.save(model.state_dict(),'73_bow_trained.pth')


Epoch0 step0 loss=0.6931
Epoch0 step320 loss=0.6902
Epoch0 step640 loss=0.6862
Epoch0 step960 loss=0.6780
Epoch0 step1280 loss=0.6565
Epoch0 step1600 loss=0.6537
Epoch0 step1920 loss=0.6594
Epoch0 step2240 loss=0.6710
Epoch0 step2560 loss=0.6664
Epoch0 step2880 loss=0.6542
Epoch0 step3200 loss=0.6630
Epoch0 step3520 loss=0.6566
Epoch0 step3840 loss=0.6154
Epoch0 step4160 loss=0.6180
Epoch0 step4480 loss=0.5974
Epoch0 step4800 loss=0.6184
Epoch0 step5120 loss=0.5950
Epoch0 step5440 loss=0.5993
Epoch0 step5760 loss=0.6107
Epoch0 step6080 loss=0.6202
Epoch0 step6400 loss=0.6309
Epoch0 step6720 loss=0.5963
Epoch0 step7040 loss=0.5507
Epoch0 step7360 loss=0.5761
Epoch0 step7680 loss=0.6147
Epoch0 step8000 loss=0.6073
Epoch0 step8320 loss=0.5521
Epoch0 step8640 loss=0.5839
Epoch0 step8960 loss=0.6060
Epoch0 step9280 loss=0.5484
Epoch0 step9600 loss=0.5489
Epoch0 step9920 loss=0.5581
Epoch0 step10240 loss=0.5622
Epoch0 step10560 loss=0.5530
Epoch0 step10880 loss=0.5847
Epoch0 step11200 loss=0

In [3]:
import numpy as np
import torch
import torch.nn as nn
import pickle
from torch.nn.utils.rnn import pad_sequence

# データと埋め込みのロード
embedding_matrix = np.load('70_embeddings.npy')
with open('71_sst_data.pkl', 'rb') as f:
    data = pickle.load(f)
    train_data = data['train']

# バッチ作成
def make_batch(batch):
    x = pad_sequence([ex['input_ids'] for ex in batch], batch_first=True, padding_value=0)
    y = torch.cat([ex['label'] for ex in batch]).squeeze()
    return x, y

# モデル定義（埋め込み固定）
class BoWClassifier(nn.Module):
    def __init__(self, embedding_matrix):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float), freeze=True, padding_idx=0)
        emb_dim = embedding_matrix.shape[1]
        self.weight = nn.Parameter(torch.zeros(emb_dim))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, input_ids):
        emb = self.embedding(input_ids)
        mask = (input_ids != 0).unsqueeze(-1).float()
        summed = (emb * mask).sum(dim=1)
        lengths = mask.sum(dim=1).clamp(min=1)
        avg = summed / lengths
        logits = avg.matmul(self.weight) + self.bias
        return torch.sigmoid(logits)

if __name__ == '__main__':
    model = BoWClassifier(embedding_matrix)
    optimizer = torch.optim.Adam([model.weight, model.bias], lr=1e-3)
    criterion = nn.BCELoss()

    model.train()
    for epoch in range(5):
        for i in range(0, len(train_data), 32):
            batch = train_data[i:i+32]
            x, y = make_batch(batch)
            optimizer.zero_grad()
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()
            if i % 320 == 0:
                print(f"Epoch {epoch}, step {i}, loss={loss.item():.4f}")

    torch.save(model.state_dict(), '73_bow_trained.pth')
    print('knock73 completed.')

Epoch 0, step 0, loss=0.6931
Epoch 0, step 320, loss=0.6902
Epoch 0, step 640, loss=0.6862
Epoch 0, step 960, loss=0.6780
Epoch 0, step 1280, loss=0.6565
Epoch 0, step 1600, loss=0.6537
Epoch 0, step 1920, loss=0.6594
Epoch 0, step 2240, loss=0.6710
Epoch 0, step 2560, loss=0.6664
Epoch 0, step 2880, loss=0.6542
Epoch 0, step 3200, loss=0.6630
Epoch 0, step 3520, loss=0.6566
Epoch 0, step 3840, loss=0.6154
Epoch 0, step 4160, loss=0.6180
Epoch 0, step 4480, loss=0.5974
Epoch 0, step 4800, loss=0.6184
Epoch 0, step 5120, loss=0.5950
Epoch 0, step 5440, loss=0.5993
Epoch 0, step 5760, loss=0.6107
Epoch 0, step 6080, loss=0.6202
Epoch 0, step 6400, loss=0.6309
Epoch 0, step 6720, loss=0.5963
Epoch 0, step 7040, loss=0.5507
Epoch 0, step 7360, loss=0.5761
Epoch 0, step 7680, loss=0.6147
Epoch 0, step 8000, loss=0.6073
Epoch 0, step 8320, loss=0.5521
Epoch 0, step 8640, loss=0.5839
Epoch 0, step 8960, loss=0.6060
Epoch 0, step 9280, loss=0.5484
Epoch 0, step 9600, loss=0.5489
Epoch 0, step 